# EDA & Visualization Strategy Playbook

A general method for tackling *any* "explore this dataset and find insights"
project — extracted from working through the Citi Bike project, but written to
generalize to other event-log-style datasets (ride shares, transit swipes,
gym check-ins, sensor logs, e-commerce orders, etc).

Use this as a checklist, not a script — skip steps that don't apply, but don't
skip them silently.

## Step 1 — Understand the grain and the schema before touching code

Answer these in a sentence each, in a markdown cell, before writing any code:
- **Grain**: what does one row represent? (one bike trip, one order, one sensor
  reading?)
- **Time span & size**: what date range, how many rows — does it fit comfortably
  in memory, or do you need a subset while iterating?
- **Keys**: is there an ID that lets you join this table to another one
  (station id, user id, order id)?
- **The business question**: who would read this analysis, and what decision
  would they make differently based on what you find? Write this down even if
  it's a guess — it stops you from producing charts nobody asked for.

## Step 2 — Load small, inspect first, scale up later

Always load a subset (or `nrow` cap) first if the file is large. Before any
transformation, run:
- `str()` / `dtypes` — confirm column types match what you'd expect.
- `head()` — eyeball a few real rows.
- Missing-value counts per column.
- Range/min/max on any numeric column you plan to use in a formula (duration,
  price, distance) — outliers in raw units usually reveal themselves here.

**Rule of thumb**: never engineer a feature from a column you haven't range-
checked.

In [ ]:
# Generic audit template (adapt column names)
# str(df)
# colSums(is.na(df))
# sapply(df[c("col_a", "col_b")], range, na.rm = TRUE)

## Step 3 — Engineer the smallest set of features that answers the question

Common transformations that show up across almost every "event log" dataset:
- **Time features**: parse a timestamp, then derive hour/day-of-week/weekend
  flag/month/season as needed by the question.
- **Age/tenure from a birth year or signup date**: `reference_year - birth_year`,
  being explicit about *which* reference year (data collection year, not
  "today").
- **Distance/duration ratios**: speed, price-per-unit, rate — anything that
  normalizes a raw total into a comparable-across-rows metric.
- **Category cleanup**: cast numeric codes that are really categories
  (gender codes, status codes, region codes) to factors/categoricals *before*
  plotting or grouping by them.

After each new column, print `head()` again. Catching a broken feature at row
6 is much cheaper than catching it in a chart 10 cells later.

## Step 4 — Start with one exploratory chart per natural grouping variable

Before building the "hero" chart the project is aiming for, make one quick,
throwaway chart per categorical or ordinal variable you have (age, hour,
weekday, category, region). You're looking for:
- A variable with a clear, monotonic, or clearly bimodal pattern (worth a
  full chart).
- A variable that's flat/noisy (probably not worth a section).
- A variable with a data-quality problem hiding in it (values at 0, one
  category dominating 90% of rows).

This triage step is what separates "I made every chart I could think of" from
"I found the two charts that matter."

## Step 5 — Build the main chart, then immediately stress-test it

For your primary chart:
1. Make the ugly version first (no labels, no filtering) just to see the
   shape.
2. Look for outliers that are probably data errors (age 140, negative price,
   0-second duration) — filter them and note *why* in a markdown cell.
3. Only then add titles, axis labels, and legend cleanup — polish is the last
   step, not the first.
4. Ask: "if my one key assumption were wrong, would this conclusion survive?"
   (For Citi Bike: if the distance formula is off by a fixed multiplier, does
   the age trend still hold? Usually the *shape* of a relationship is more
   robust than its *scale*.)

## Step 6 — Add one cross-cut the original prompt didn't ask for

A single grouping variable (age) is a good start, but a second cross-cut
(age × gender, age × user type, region × time-of-day) is usually where the
more interesting, decision-relevant finding lives. Before doing this, check:
- Do you have enough data in each combined group to trust the summary stat?
  (`group_by(a, b) %>% tally()` and eyeball the smallest groups.)
- Does adding the second dimension change the *interpretation*, or just add
  visual noise? If it's just noise, say so and move on — not every cross-cut
  is worth a slide.

## Step 7 — Look for an operational/structural angle, not just a demographic one

Demographic breakdowns (age, gender) answer "who." Datasets like this usually
also support a structural question that's more directly actionable:
- **Flow/imbalance**: for anything with a "from" and "to" (station, branch,
  warehouse), compute in-flow minus out-flow per node to find bottlenecks.
- **Temporal load**: counts by hour/day to find peak-demand windows worth
  staffing or capacity decisions around.
- **Segment comparison**: a categorical split baked into the data (subscriber
  vs. casual, new vs. returning, paid vs. free) that maps directly onto a
  business lever.

These tend to be more useful to a real stakeholder than a fourth demographic
chart.

## Step 8 — Name your assumptions and your confidence, in writing

Close every project with a short written section (not just charts) that states:
1. The headline finding, in one sentence a non-technical stakeholder could
   repeat.
2. The 2-3 numbers/charts that back it up.
3. The single assumption that, if wrong, would most change the conclusion.
4. The one additional piece of data or analysis you'd want next.

This four-part memo format works for almost any exploratory project and is
usually what turns an exercise into something you can put in a portfolio or
hand to a stakeholder.

## Quick self-check before calling a project "done"

- [ ] Did I check missing values and implausible ranges before modeling?
- [ ] Did I cast every "numeric but really categorical" column to a factor?
- [ ] Does every chart have a title, axis labels, and (if needed) a legible legend?
- [ ] Did I filter outliers *and say why* rather than silently dropping them?
- [ ] Did I add at least one cross-cut or structural angle beyond the obvious one?
- [ ] Did I write a short takeaway that names my biggest assumption?
- [ ] Could someone else re-run this notebook top to bottom without errors?